In [1]:
import os

In [2]:
import numpy as np

In [3]:
import pandas as pd

In [4]:
import matplotlib.pyplot as plt

In [5]:
from PIL import Image

In [6]:
dataset_path = "Dataset_BUSI_with_gt"
print(os.listdir(dataset_path))

['benign', 'malignant', 'normal']


In [7]:
classes = ["benign","malignant","normal"]

In [8]:
for category in classes:
    folder_path = os.path.join(dataset_path,category)
    files = os.listdir(folder_path)
    images = [file for file in files if file.lower().endswith((".png",".jpg",".jpeg")) and "_mask" not in file.lower()]
    print(category,":", len(images))

benign : 437
malignant : 210
normal : 133


In [9]:
import os
for category in classes:
    folder_path = os.path.join(dataset_path,category)
    files = os.listdir(folder_path)
    print("\nCategory:",category)
    for file in files[:10]:
        print(file)


Category: benign
benign (1).png
benign (1)_mask.png
benign (10).png
benign (10)_mask.png
benign (100).png
benign (100)_mask.png
benign (100)_mask_1.png
benign (101).png
benign (101)_mask.png
benign (102).png

Category: malignant
malignant (1).png
malignant (1)_mask.png
malignant (10).png
malignant (10)_mask.png
malignant (100).png
malignant (100)_mask.png
malignant (101).png
malignant (101)_mask.png
malignant (102).png
malignant (102)_mask.png

Category: normal
normal (1).png
normal (1)_mask.png
normal (10).png
normal (10)_mask.png
normal (100).png
normal (100)_mask.png
normal (101).png
normal (101)_mask.png
normal (102).png
normal (102)_mask.png


In [10]:
import os 
import hashlib
from collections import defaultdict

In [11]:
import os 
import hashlib 
from collections import defaultdict

image_hashes = defaultdict(list)
total_images = 0
for category in classes:
    folder_path = os.path.join(dataset_path,category)
    for file in os.listdir(folder_path):
        if file.lower().endswith((".png",".jpg",".jpeg")):
            if "_mask" in file.lower():
                continue
            image_path = os.path.join(folder_path,file)
            with open(image_path , "rb") as f:
                image_hash = hashlib.sha256(f.read()).hexdigest()
            image_hashes[image_hash].append({"path": image_path,"category":category})
            total_images +=1

# find the duplicates group
duplicates = {key:value for key,value in image_hashes.items() if len(value) > 1}

#print result after processing all images
print("Total original images:",total_images)
print("unique image hashes :",len(image_hashes))
print("Duplicates groups:",len(duplicates))

#Display duplicates information
for group in duplicates.values():
    print("\nDuplicate group:")
    for image in group:
        print(image["category"],":",image["path"])

Total original images: 780
unique image hashes : 779
Duplicates groups: 1

Duplicate group:
benign : Dataset_BUSI_with_gt\benign\benign (433).png
malignant : Dataset_BUSI_with_gt\malignant\malignant (145).png


In [12]:
conflicting_hashes = {key for key,group in image_hashes.items() if len(set(item["category"] for item in group)) > 1}

# create clean dataset record
clean_records = []
for key,group in image_hashes.items():
    if key in conflicting_hashes:
        continue
    # key one copy of each exact image 
    clean_records.append(group[0])
#Display clean dataset information 
print("clean images:",len(clean_records))
for category in classes:
    print(category,":", sum(item["category"] == category for item in clean_records))
print("conflicting duplicate groups:",len(conflicting_hashes))


clean images: 778
benign : 436
malignant : 209
normal : 133
conflicting duplicate groups: 1


In [13]:
print("Total original images:",total_images)
print("Total unique hashes:",len(image_hashes))
print("Total clean records:",len(clean_records))
print("\nImages in each category:") 
for category in classes:
    count =sum(1 for group in image_hashes.values() for item in group if item["category"] == category)
    print(category,":",count)

Total original images: 780
Total unique hashes: 779
Total clean records: 778

Images in each category:
benign : 437
malignant : 210
normal : 133


In [14]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.DataFrame(clean_records)   # columns: path, category

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["category"], random_state=42)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["category"], random_state=42)

# sanity checks
assert not set(train_df.path) & set(val_df.path)
assert not set(train_df.path) & set(test_df.path)
assert not set(val_df.path) & set(test_df.path)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), d["category"].value_counts(normalize=True).round(2).to_dict())

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

train 544 {'benign': 0.56, 'malignant': 0.27, 'normal': 0.17}
val 117 {'benign': 0.56, 'malignant': 0.27, 'normal': 0.17}
test 117 {'benign': 0.56, 'malignant': 0.26, 'normal': 0.17}


In [16]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH = 16
AUTOTUNE = tf.data.AUTOTUNE

classes = ["benign", "malignant", "normal"]
class_to_idx = {c: i for i, c in enumerate(classes)}

train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("val.csv")
test_df  = pd.read_csv("test.csv")

In [17]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_ds(df, training=False):
    paths = df["path"].values
    labels = df["category"].map(class_to_idx).values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if training:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

In [18]:
from sklearn.utils.class_weight import compute_class_weight

y_train = train_df["category"].map(class_to_idx).values
weights = compute_class_weight("balanced", classes=np.arange(3), y=y_train)
class_weight = dict(enumerate(weights))
print(class_weight)

{0: np.float64(0.594535519125683), 1: np.float64(1.2420091324200913), 2: np.float64(1.949820788530466)}


In [19]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

base = keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = augment(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation="softmax")(x)
model = keras.Model(inputs, outputs)

In [20]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

callbacks1 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
    keras.callbacks.ModelCheckpoint("stage1.keras", save_best_only=True),
]

history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=15, class_weight=class_weight, callbacks=callbacks1)

Epoch 1/15


C:\Users\pavan\anaconda3\envs\tensorflow\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


34/34 ━━━━━━━━━━━━━━━━━━━━ 49s 837ms/step - accuracy: 0.3989 - loss: 1.0889 - val_accuracy: 0.5983 - val_loss: 0.8951 - learning_rate: 0.0010
Epoch 2/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 31s 579ms/step - accuracy: 0.5662 - loss: 0.8651 - val_accuracy: 0.6325 - val_loss: 0.8221 - learning_rate: 0.0010
Epoch 3/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 19s 562ms/step - accuracy: 0.6360 - loss: 0.7794 - val_accuracy: 0.6667 - val_loss: 0.7809 - learning_rate: 0.0010
Epoch 4/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 543ms/step - accuracy: 0.6691 - loss: 0.7210 - val_accuracy: 0.7179 - val_loss: 0.7347 - learning_rate: 0.0010
Epoch 5/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 20s 535ms/step - accuracy: 0.6838 - loss: 0.6903 - val_accuracy: 0.7350 - val_loss: 0.6969 - learning_rate: 0.0010
Epoch 6/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 17s 487ms/step - accuracy: 0.6985 - loss: 0.6789 - val_accuracy: 0.6752 - val_loss: 0.7187 - learning_rate: 0.0010
Epoch 7/15
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 520ms/step - accuracy: 0.7022 - loss: 0.6582 - val_

In [21]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

callbacks2 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
    keras.callbacks.ModelCheckpoint("stage2.keras", save_best_only=True),
]

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=20, class_weight=class_weight, callbacks=callbacks2)

Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 55s 939ms/step - accuracy: 0.7445 - loss: 0.6043 - val_accuracy: 0.7094 - val_loss: 0.6424 - learning_rate: 1.0000e-05
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 32s 660ms/step - accuracy: 0.7555 - loss: 0.5770 - val_accuracy: 0.7094 - val_loss: 0.6396 - learning_rate: 1.0000e-05
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 42s 680ms/step - accuracy: 0.7353 - loss: 0.5738 - val_accuracy: 0.7179 - val_loss: 0.6206 - learning_rate: 1.0000e-05
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 20s 574ms/step - accuracy: 0.7463 - loss: 0.5704 - val_accuracy: 0.7179 - val_loss: 0.6260 - learning_rate: 1.0000e-05
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 643ms/step - accuracy: 0.7684 - loss: 0.5341 - val_accuracy: 0.7265 - val_loss: 0.6131 - learning_rate: 1.0000e-05
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 20s 593ms/step - accuracy: 0.7482 - loss: 0.5346 - val_accuracy: 0.7350 - val_loss: 0.6042 - learning_rate: 1.0000e-05
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 23s 682ms/step - acc

In [22]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

probs = model.predict(val_ds)
y_pred = probs.argmax(axis=1)
y_true = np.concatenate([y.numpy() for _, y in val_ds])

print(classification_report(y_true, y_pred, target_names=classes, digits=3))
print("Macro ROC-AUC:", roc_auc_score(y_true, probs, multi_class="ovr"))
print(confusion_matrix(y_true, y_pred))

8/8 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step
              precision    recall  f1-score   support

      benign      0.865     0.692     0.769        65
   malignant      0.605     0.719     0.657        32
      normal      0.704     0.950     0.809        20

    accuracy                          0.744       117
   macro avg      0.725     0.787     0.745       117
weighted avg      0.767     0.744     0.745       117

Macro ROC-AUC: 0.9051697699759463
[[45 15  5]
 [ 6 23  3]
 [ 1  0 19]]


In [23]:
model.save("stage2_v1.keras")

base.trainable = True
for layer in base.layers[:-80]:            # unfreeze more layers (was 30)
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),   # 10x higher (was 1e-5)
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

callbacks3 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
    keras.callbacks.ModelCheckpoint("stage2_v2.keras", save_best_only=True),
]

history3 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=30, class_weight=class_weight, callbacks=callbacks3)

Epoch 1/30


C:\Users\pavan\anaconda3\envs\tensorflow\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


34/34 ━━━━━━━━━━━━━━━━━━━━ 50s 811ms/step - accuracy: 0.7941 - loss: 0.5000 - val_accuracy: 0.7009 - val_loss: 0.6556 - learning_rate: 1.0000e-04
Epoch 2/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 22s 636ms/step - accuracy: 0.7941 - loss: 0.4600 - val_accuracy: 0.7094 - val_loss: 0.5937 - learning_rate: 1.0000e-04
Epoch 3/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 610ms/step - accuracy: 0.8199 - loss: 0.4181 - val_accuracy: 0.7692 - val_loss: 0.5142 - learning_rate: 1.0000e-04
Epoch 4/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 40s 590ms/step - accuracy: 0.8143 - loss: 0.4325 - val_accuracy: 0.7094 - val_loss: 0.6092 - learning_rate: 1.0000e-04
Epoch 5/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 20s 599ms/step - accuracy: 0.8309 - loss: 0.3602 - val_accuracy: 0.7521 - val_loss: 0.6007 - learning_rate: 1.0000e-04
Epoch 6/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 20s 601ms/step - accuracy: 0.8732 - loss: 0.2898 - val_accuracy: 0.7179 - val_loss: 0.6648 - learning_rate: 1.0000e-04
Epoch 7/30
34/34 ━━━━━━━━━━━━━━━━━━━━ 20s 600ms/step - accuracy: 0.86

In [24]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

probs = model.predict(val_ds)
y_pred = probs.argmax(axis=1)
y_true = np.concatenate([y.numpy() for _, y in val_ds])

print(classification_report(y_true, y_pred, target_names=classes, digits=3))
print("Macro ROC-AUC:", roc_auc_score(y_true, probs, multi_class="ovr"))
print(confusion_matrix(y_true, y_pred))

# malignant-vs-rest threshold table
y_mal = (y_true == 1).astype(int)
p_mal = probs[:, 1]
print("Malignant-vs-rest AUC:", roc_auc_score(y_mal, p_mal))
for t in [0.5, 0.4, 0.3, 0.25, 0.2]:
    pred = p_mal >= t
    tp = (pred & (y_mal == 1)).sum()
    fn = ((~pred) & (y_mal == 1)).sum()
    fp = (pred & (y_mal == 0)).sum()
    print(f"threshold {t}: recall {tp/(tp+fn):.3f}  precision {tp/max(tp+fp,1):.3f}  missed {fn}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 967ms/step
              precision    recall  f1-score   support

      benign      0.873     0.738     0.800        65
   malignant      0.605     0.812     0.693        32
      normal      0.842     0.800     0.821        20

    accuracy                          0.769       117
   macro avg      0.773     0.784     0.771       117
weighted avg      0.794     0.769     0.774       117

Macro ROC-AUC: 0.9171471069669862
[[48 16  1]
 [ 4 26  2]
 [ 3  1 16]]
Malignant-vs-rest AUC: 0.8805147058823529
threshold 0.5: recall 0.812  precision 0.619  missed 6
threshold 0.4: recall 0.875  precision 0.596  missed 4
threshold 0.3: recall 0.875  precision 0.528  missed 4
threshold 0.25: recall 0.875  precision 0.528  missed 4
threshold 0.2: recall 0.906  precision 0.500  missed 3


In [25]:
model.save("best_stage2_v2.keras")

THRESH = 0.4
def predict_with_threshold(probs, t=THRESH):
    pred = probs.argmax(axis=1)
    pred[probs[:, 1] >= t] = 1
    return pred

probs_t = model.predict(test_ds)
y_true_t = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_t = predict_with_threshold(probs_t)

print(classification_report(y_true_t, y_pred_t, target_names=classes, digits=3))
print(confusion_matrix(y_true_t, y_pred_t))

8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 357ms/step
              precision    recall  f1-score   support

      benign      0.889     0.727     0.800        66
   malignant      0.643     0.871     0.740        31
      normal      0.857     0.900     0.878        20

    accuracy                          0.795       117
   macro avg      0.796     0.833     0.806       117
weighted avg      0.818     0.795     0.797       117

[[48 15  3]
 [ 4 27  0]
 [ 2  0 18]]


In [26]:
grad_model = keras.Model(base.inputs,
    [base.get_layer("top_activation").output, base.output])
gap_layer, dense_layer = model.layers[-3], model.layers[-1]

def gradcam(img_batch, class_idx=None):
    with tf.GradientTape() as tape:
        conv, feat = grad_model(img_batch)
        preds = dense_layer(gap_layer(feat))
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), preds.numpy()[0]

In [27]:
import json
from sklearn.metrics import classification_report, confusion_matrix

report = classification_report(y_true_t, y_pred_t, target_names=classes, output_dict=True)
json.dump({"report": report,
           "confusion_matrix": confusion_matrix(y_true_t, y_pred_t).tolist(),
           "threshold": 0.4}, open("metrics.json", "w"), indent=2)

In [1]:
%pip uninstall pyarrow -y

Found existing installation: pyarrow 26.0.1
Uninstalling pyarrow-26.0.1:
  Successfully uninstalled pyarrow-26.0.1
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


In [1]:
import tensorflow as tf, keras
print(tf.__version__, keras.__version__)

2.21.0 3.15.1
